# Hypothetical Document Embeddings (HyDE)

Hypothetical Document Embeddings (HyDE) addresses the vocabulary and stylistic mismatch between brief user queries and rich reference documents. Using an instruction-following Groq LLM (`openai/gpt-oss-120b`), HyDE first generates a hypothetical answer, embeds that hallucinated document using HuggingFace MiniLM, and queries the vector store for actual factual documents inhabiting that semantic neighborhood.

## Workflow Architecture

<div align="center">
  <img src="workflow_hyde.png" alt="Hypothetical Document Embeddings (HyDE) Architecture Diagram" width="580" />
</div>

<details>
<summary><b>Click to expand Colorful Mermaid Source Code</b></summary>

```mermaid
flowchart TD
    subgraph In["User Query"]
        Q(["1. Original Query"]):::startNode
    end
    subgraph Gen["Hypothetical Generation"]
        LLM["2. LLM Generator<br/><b>Groq gpt-oss-120b</b><br/>(Hallucinates Hypothetical Document)"]:::llmNode
        HypoDoc["3. Hypothetical Document<br/>(Captures Answer Structure)"]:::docNode
    end
    subgraph Ret["Semantic Retrieval"]
        Emb["4. Embedding Model<br/><b>HuggingFace MiniLM</b><br/>(Embeds Hypothetical Document)"]:::embNode
        VS["5. FAISS Vector Store<br/>(Cosine Nearest Neighbor Search)"]:::vsNode
        RealDocs["6. Retrieved Ground Truth Documents"]:::realDocNode
    end
    subgraph Out["Final Synthesis"]
        Synth["7. Final LLM Synthesis<br/>(Answers Query with Retrieved Facts)"]:::synthNode
        Ans(["8. Grounded Truth Answer"]):::endNode
    end
    Q --> LLM
    LLM --> HypoDoc
    HypoDoc --> Emb
    Emb --> VS
    VS --> RealDocs
    Q --> Synth
    RealDocs --> Synth
    Synth --> Ans
    classDef startNode fill:#E8F5E9,stroke:#2E7D32,stroke-width:2px,color:#1B5E20;
    classDef llmNode fill:#E3F2FD,stroke:#1565C0,stroke-width:2px,color:#0D47A1;
    classDef docNode fill:#FFF8E1,stroke:#FFA000,stroke-width:2px,color:#E65100;
    classDef embNode fill:#EDE7F6,stroke:#5E35B1,stroke-width:2px,color:#311B92;
    classDef vsNode fill:#E0F7FA,stroke:#00838F,stroke-width:2px,color:#004D40;
    classDef realDocNode fill:#E8EAF6,stroke:#3F51B5,stroke-width:2px,color:#1A237E;
    classDef synthNode fill:#FCE4EC,stroke:#C2185B,stroke-width:2px,color:#880E4F;
    classDef endNode fill:#FFEBEE,stroke:#D32F2F,stroke-width:2px,color:#B71C1C;
```
</details>

### Key Retrieval Principles
- **Hypothetical Answer Formulation**: Synthesizes speculative text capturing the linguistic structure of the true target passage.
- **Semantic Neighborhood Search**: HuggingFace MiniLM embeds the hypothetical text to locate authentic matching passages.
- **Zero-Shot Relevance Boosting**: 显著 increases retrieval precision on questions where standard query keywords are sparse.


In [4]:
import datetime
import wikipedia
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

# Configure Wikipedia API user-agent and rate limiting to prevent HTTP 429 / JSONDecodeError
wikipedia.set_user_agent("MyHydeRagApp/1.0 (contact: student@example.com)")
wikipedia.set_rate_limiting(True, min_wait=datetime.timedelta(seconds=1))

chunk_size = 300
chunk_overlap = 100
loader = WikipediaLoader(query="Steve Jobs", load_max_docs=5)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
docs = text_splitter.split_documents(documents=documents)


In [10]:
from langchain.chat_models import init_chat_model
from langchain_community.vectorstores import Chroma

embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

llm = init_chat_model(model="groq:openai/gpt-oss-120b")

db = Chroma.from_documents(documents=docs, embedding=embedding, persist_directory="output/steve_jobs_for_hyde.db")
base_retriver = db.as_retriever(search_kwargs={'k': 5})


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12209.97it/s]


In [ ]:
from google.genai._gaos.utils import queryparams
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate


def get_hyde_doc(query):
    template = """Imagine you are an expert writing a detailed explanation on the topic: '{query}'
    create a hypothetical answer for the topic"""
    system_message_prompt = SystemMessagePromptTemplate.from_template(template=template)
    chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt])
    messages = chat_prompt.format_prompt(query=query).to_messages() # Internally, the {query} placeholder is replaced
    response = llm.invoke(messages)
    print(response)
    return response.content

In [13]:
query = "When was Steve Jobs fired from Apple?"
matched_doc = base_retriver.invoke(get_hyde_doc(query))
print(matched_doc)

content='**When Was Steve\u202fJobs Fired from Apple? – A Detailed Timeline and Contextual Overview**\n\n---\n\n### The Quick Answer\nSteve\u202fJobs was effectively forced out of Apple on **July\u202f9\u202f1997**, when the company’s board of directors accepted his resignation as an “interim” CEO and appointed former IBM executive **Gil Amelio** as the permanent chief executive. However, the events that led to his ouster began much earlier, and the final “firing” was the culmination of a power struggle that started in the mid‑1990s.\n\n---\n\n## 1. Background: How Jobs Came to Power at Apple\n\n| Year | Milestone | Significance |\n|------|-----------|--------------|\n| **1976** | Co‑founding of Apple Computer, Inc. with Steve Wozniak and Ronald Wayne | Launched the personal computer revolution with the Apple I. |\n| **1977–1984** | Development of the Apple II, Lisa, and Macintosh | Established Apple as an innovative tech leader. |\n| **1985** | Jobs becomes Chairman of the Board and “

In [ ]:
from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder
from langchain_community.document_loaders import TextLoader


loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)
base_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# Step 3: HyDE Embedder using a built-in prompt template
hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(llm=llm,base_embeddings=base_embeddings,prompt_key="web_search")

# or Custom HyDE Prompt
# custom = PromptTemplate.from_template(
# "Generate a concise hypothetical answer for this topic: {query}"
# )
# hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
# llm=llm,
# base_embeddings=base_embeddings,
# custom_prompt=custom
# )

vectorstore = Chroma.from_documents(documents=chunks,embedding=hyde_embedding_function,persist_directory="output/langchain")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9960.65it/s]


In [15]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate

rag_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.
Context:
{context}
Question: {input}
""")
rag_chain = create_stuff_documents_chain(llm=llm, prompt=rag_prompt)

def hyde_rag_pipeline(query):
    matched_docs = vectorstore.similarity_search(query, k=4)
    response = rag_chain.invoke({"input": query, "context": matched_docs})
    return response

query = "What memory modules does LangChain provide?"
answer = hyde_rag_pipeline(query)
print("Final Answer:\n", answer)

Final Answer:
 LangChain includes memory modules such as:

- **ConversationBufferMemory** – keeps a running buffer of all prior conversation turns so the model can reference the full dialogue history.  
- **ConversationSummaryMemory** – generates and stores a summary of the conversation, allowing the model to retain the gist of long interactions while staying within token limits.


Custom HyDE Prompt

In [ ]:
custom = PromptTemplate.from_template(
"Generate a concise hypothetical answer for this topic: {query}"
)
hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
llm=llm,
base_embeddings=base_embeddings,
custom_prompt=custom
)